In [ ]:
_NB_NAME_="EDA & Data Profiling"
# @title 🤖 AI Grading
# @markdown **Enter your GitHub username, then click ▶ Run.**
# @markdown Colab will send your answers to Gemini automatically — no key needed.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

import textwrap
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {v or '(no answer)'}"
                    for i,(_,v) in enumerate(answers.items(),1))

    prompt = (f"Grade these quiz answers for the \"{_NB_TITLE}\" notebook "
              f"(Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
              f"{qa}\n\n"
              f"For each answer say CORRECT, PARTIAL, or INCORRECT and one sentence why. "
              f"Accept any reasonable paraphrase as correct. "
              f"End with an overall grade (Excellent/Good/Needs Review/Incomplete) "
              f"and one sentence on what to review if they struggled.")

    # Try Colab's built-in Gemini via the generativeai SDK (no key needed in Colab)
    success = False
    try:
        import google.generativeai as genai
        # In Colab, calling generate_content without configure() uses
        # Colab's own managed credentials automatically
        model = genai.GenerativeModel("gemini-1.5-flash")
        resp  = model.generate_content(prompt)
        print("\n" + "\u2550"*56)
        print(f"  \U0001f916 Feedback \u2014 {_NB_TITLE}")
        if GITHUB_USERNAME.strip():
            print(f"  Student: @{GITHUB_USERNAME.strip()}")
        print("\u2550"*56)
        for line in resp.text.strip().split("\n"):
            if line.strip():
                for w in textwrap.wrap(line.strip(), 54):
                    print(f"  {w}")
            else:
                print()
        print("\u2550"*56)
        success = True
    except Exception:
        pass

    if not success:
        # Fallback: print the prompt so they can paste it into Colab's AI chat
        print("\u2550"*56)
        print("  Automatic grading unavailable.")
        print("  Paste the text below into the Gemini chat panel:")
        print("  (click the \u2728 sparkle icon in the Colab toolbar)")
        print("\u2550"*56)
        print()
        print(prompt)
        print()
        print("\u2550"*56)

    print("\n  \U0001f4be File \u2192 Save a copy in GitHub \u2192 your fork")


# 🔍 EDA & Data Profiling
**ISLP Supplement · Pattern Recognition for the Rest of Us**

> Exploratory Data Analysis is the first thing you do with any new dataset — before modeling, before cleaning, before anything. The goal is to understand the data well enough to make good decisions about everything that follows.

---
### What you'll learn
- A structured EDA workflow you can apply to any dataset
- Automated profiling with `ydata-profiling` — full report in one line
- Distributions, missing data, outliers, and correlations
- How EDA informs modeling decisions

### Dataset
We'll use the **Titanic** passenger dataset — survival prediction. 891 passengers, mix of numeric and categorical features.

### Time
~60 minutes

In [ ]:
# Titanic via seaborn (bundled, no network needed in Colab)
import seaborn as sns
titanic = sns.load_dataset('titanic')
# Clean up
titanic['sex_num'] = (titanic['sex'] == 'female').astype(int)
titanic = titanic[['survived','pclass','sex_num','age','sibsp','parch','fare']].dropna()
print(f"Titanic: {titanic.shape}  |  Survival rate: {titanic['survived'].mean():.1%}")
titanic.head()

## 📋 Part 1 — First Look: Shape, Types, Missing Values

Always start with these three questions:
1. **How many rows and columns?** (scale of the problem)
2. **What data types are present?** (numeric, categorical, datetime, text)
3. **Where is data missing?** (missing data = decisions to make)

In [ ]:
# Step 1: Basic overview
print("=== Shape ===")
print(f"{df.shape[0]} passengers × {df.shape[1]} features\n")

print("=== Data Types ===")
print(df.dtypes.to_string())

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
print(missing_df.to_string())

In [ ]:
# Visualize missing data
fig, ax = plt.subplots(figsize=(10, 3))
missing_pct_all = (df.isnull().sum() / len(df) * 100)
colors = ['#e85d20' if v > 20 else '#1e5fa8' if v > 0 else '#d0d8e8' 
          for v in missing_pct_all]
bars = ax.bar(missing_pct_all.index, missing_pct_all.values, 
              color=colors, edgecolor='white')
ax.set_ylabel('Missing (%)')
ax.set_title('Missing Data by Column — Orange = >20% missing (consider dropping)')
ax.set_xticklabels(missing_pct_all.index, rotation=45, ha='right')

for bar, val in zip(bars, missing_pct_all):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print("\n📌 Age: 19.9% missing — impute (median/mean) or use a missing indicator")
print("   Cabin: 77.1% missing — too many missing to impute reliably, consider dropping")
print("   Embarked: 0.2% missing — safe to impute with mode")

## 📊 Part 2 — Distributions: Numeric Features

For each numeric feature, check:
- **Shape**: normal, skewed, bimodal?
- **Range**: any impossible values?
- **Outliers**: points far from the bulk of the data

In [ ]:
# Distributions of all numeric features
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols[:8]):
    data = df[col].dropna()
    axes[i].hist(data, bins=30, color='#1e5fa8', edgecolor='white', alpha=0.8)
    axes[i].axvline(data.mean(),   color='#e85d20', lw=2, label=f'Mean={data.mean():.1f}')
    axes[i].axvline(data.median(), color='#1a7a45', lw=2, ls='--', label=f'Median={data.median():.1f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)
    skew = data.skew()
    axes[i].set_xlabel(f'Skewness: {skew:.2f}')

for j in range(len(numeric_cols), 8):
    axes[j].set_visible(False)

plt.suptitle('Distributions of Numeric Features', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("\n📌 Fare is highly right-skewed (skew > 2) — log transform often helps")
print("   Age is roughly normal with slight right skew")
print("   SibSp and Parch are heavily zero-inflated — most passengers traveled alone")

In [ ]:
# Box plots to spot outliers
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, color in zip(axes, ['Age','Fare','SibSp'], ['#1e5fa8','#e85d20','#6b46c1']):
    bp = ax.boxplot(df[col].dropna(), patch_artist=True,
                   boxprops=dict(facecolor=color, alpha=0.6),
                   medianprops=dict(color='black', lw=2),
                   flierprops=dict(marker='o', markerfacecolor=color, markersize=4, alpha=0.5))
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)][col]
    ax.set_title(f'{col}\n{len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')
    ax.set_ylabel(col)

plt.suptitle('Outlier Detection via Box Plots', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Fare has extreme outliers — some first-class passengers paid 10x the median")

## 🎨 Part 3 — Categorical Features & Target Analysis

For categorical features:
- Value counts and proportions
- Survival rate *per category* — this is the start of feature analysis

In [ ]:
# Categorical feature analysis with survival breakdown
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
cat_features = ['Survived','Pclass','Sex','Embarked','SibSp','Parch']
colors_cat = ['#1a7a45','#e85d20']

for ax, col in zip(axes.flatten(), cat_features):
    if col == 'Survived':
        counts = df[col].value_counts()
        ax.bar(['Did not survive (0)','Survived (1)'], counts.values, 
               color=['#e85d20','#1a7a45'], edgecolor='white')
        ax.set_title(f'Target: Survived\nSurvival rate: {df["Survived"].mean():.1%}')
    else:
        survival_rate = df.groupby(col)['Survived'].mean().sort_index()
        count = df[col].value_counts().sort_index()
        ax2 = ax.twinx()
        ax.bar(survival_rate.index.astype(str), count.values, 
               color='#d0d8e8', edgecolor='white', label='Count')
        ax2.plot(survival_rate.index.astype(str), survival_rate.values, 
                 'o-', color='#1e5fa8', lw=2, markersize=7, label='Survival rate')
        ax2.set_ylabel('Survival Rate', color='#1e5fa8')
        ax2.set_ylim(0, 1)
        ax2.tick_params(axis='y', labelcolor='#1e5fa8')
        ax.set_title(col)
        ax.set_xlabel(col)
        ax.set_ylabel('Count')

plt.suptitle('Categorical Features vs Survival Rate', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 Key findings:")
print("   Sex: women survived at 74%, men at 19% — strongest predictor")
print("   Pclass: 1st class 63%, 2nd class 47%, 3rd class 24%")
print("   Embarked: modest effect — may proxy for class/cabin location")

## 🔗 Part 4 — Correlations

Correlation measures linear association between numeric features. Values range from -1 to +1.

⚠️ Correlation ≠ causation. Always inspect visually.

In [ ]:
# Correlation heatmap
numeric_df = df[['Survived','Pclass','Age','SibSp','Parch','Fare']].copy()
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # upper triangle only
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        val = corr.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=10, color='white' if abs(val) > 0.5 else 'black',
                fontweight='bold' if abs(val) > 0.3 else 'normal')

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.index)
ax.set_title('Correlation Matrix — Titanic Numeric Features')
plt.tight_layout()
plt.show()

print("\n📌 Fare is positively correlated with Survived (r=0.26) — higher fare → more likely to survive")
print("   Pclass negatively correlated with Survived (r=-0.34) — higher class number = lower class = worse odds")
print("   SibSp and Parch are moderately correlated — family size features")

## ⚡ Part 5 — Automated Profiling with ydata-profiling

Manual EDA is powerful but slow. `ydata-profiling` generates a complete HTML report covering all of the above automatically.

In [ ]:
# Generate automated profiling report
# This takes ~30 seconds and generates a full interactive HTML report
profile = ProfileReport(
    df, 
    title="Titanic EDA Report",
    explorative=True,
    minimal=False  # set to True for faster but less detailed report
)

# Save to file
profile.to_file("titanic_eda_report.html")
print("✓ Report saved to titanic_eda_report.html")
print("  To view: Files panel (left sidebar) → titanic_eda_report.html → right-click → Download")
print("  Or open directly: from google.colab import files; files.download('titanic_eda_report.html')")

In [ ]:
# Display inline in Colab
from IPython.display import IFrame
profile.to_file("titanic_eda_report.html")
IFrame(src='titanic_eda_report.html', width='100%', height=600)

## 🧹 Part 6 — EDA → Decisions

EDA is not just exploration — it produces a list of *decisions* you need to make before modeling.

Based on our Titanic EDA:

In [ ]:
# Summarize EDA findings → modeling decisions
print("=" * 60)
print("EDA FINDINGS → MODELING DECISIONS")
print("=" * 60)

findings = [
    ("Missing: Age (20%)",     "Impute with median, or add 'Age_missing' flag"),
    ("Missing: Cabin (77%)",   "Drop column — too sparse to be useful"),
    ("Missing: Embarked (2)",  "Impute with mode ('S')"),
    ("Fare: right-skewed",     "Consider log1p transform before modeling"),
    ("Sex: strong predictor",  "Keep as-is, encode Male=0/Female=1"),
    ("Pclass: ordinal",        "Use as numeric (1/2/3) or one-hot encode"),
    ("SibSp + Parch",          "Consider combining into 'FamilySize' feature"),
    ("Name column",            "Extract Title (Mr/Mrs/Miss) — may carry signal"),
]

for finding, decision in findings:
    print(f"  {'Finding:':<12} {finding}")
    print(f"  {'Decision:':<12} {decision}")
    print()

print("📌 EDA doesn't just explore — it creates your data prep checklist.")

## 💼 Exercise

**Task 1:** Load a different dataset (suggestions below) and run a full EDA:
- Boston Housing: `from sklearn.datasets import fetch_california_housing`
- Diabetes: `from sklearn.datasets import load_diabetes`
- Your own CSV: `pd.read_csv('your_file.csv')`

**Task 2:** For your dataset, answer:
1. How many missing values? Where?
2. Which numeric features are skewed?
3. Which features correlate most with the target?
4. What are your top 3 modeling decisions based on EDA?

**Task 3:** Generate a ydata-profiling report for your dataset. Download and review it.

In [ ]:
# Exercise workspace
# YOUR CODE HERE — load your dataset and run EDA

In [ ]:
# @title 📝 Quiz — EDA & Data Profiling
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** Name 3 things you look for in an EDA
# @markdown **Q2:** What does a skewness value > 2 suggest about a feature?
# @markdown **Q3:** What is one limitation of correlation as a measure of association?
# @markdown **Q4:** If 40% of a column is missing, what are your options?
# @markdown **Q5:** Why should EDA come before model building, not after?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results